## Task


The aim of the assignment is to apply the NLP techniques you have learnt in class to analyse one of the 
datasets described below.   
• Note: some of the datasets are quite large, so you may need to sample a small percentage of the 
data and work that. 
The exact tasks performed may depend on the dataset chosen, but we would expect to see some of the 
following: 
#### 1. Preliminary analysis:  
Briefly describe the data:

What is the structure of the dataset? What type of task was the dataset collected for? 

What type of documents does it contain? How many are there? How long are they on average and 

what is their distribution? 

How big is the vocabulary of the collection? How big is the vocabulary of a document on average? 

Play around with documents using code from the early parts of the course. For example, you could:
Cluster the documents, visualise the clusters and to try to understand what types of groups are 
present.

Index the documents so that you can perform keyword search over them. 

Train a Word2Vec embedding and investigate the properties of the resulting embedding. 
#### 2. Training models: 
Each dataset has been created with a particular task in mind. You don’t necessarily need to tackle that 
particular problem, but you do need to train some model(s) on the data:

train ML models (e.g. a linear classifier, an LSTM and/or a Transformer) to perform a particular 

task on the data; 
if possible, try to fine-tune a pretrained models on the same task and compare their performance; 

try an LLM on the task, comparing one, few and zero-shot performance;  
and perhaps even try to fine-tune a small LLM on the task (if it makes sense to do so).   
#### 3. Possible extensions: 
Depending on the dataset chosen there will be many additional investigations that you could perform, 
for example:  
- investigate another task on the same dataset  
- investigate the same task on a related dataset 
- use text-to-speech and speech-to-text models to create a voice interactive chatbot
- create your own dialog dataset by transcribing audio conversations (e.g. using MS Teams).  

# 1. Preliminary analysis

## 1.a) Data Exploration

### Import the dataset and see how data are stored inside of it

Import the general necessary libraries ( the specific libraries will be imported later in the cells where they are used)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import random
from collections import Counter
from textblob import TextBlob
from tqdm import tqdm

In [ ]:
from datasets import load_dataset

dataset = load_dataset("neural-bridge/rag-dataset-12000")

In [ ]:
dataset

They are already divided into train and test set. So we just need to convert them to Pandas Dataframes.

In [ ]:
train_df = dataset['train'].to_pandas()
test_df = dataset['test'].to_pandas()

In [ ]:
print('Train dataset len:', len(train_df))
print('Test dataset len:', len(test_df))

Let's then print a complete (question, context, answer) sample of our dataset

In [ ]:
# Show an example

context = train_df['context'][1]
question = train_df['question'][1]
answer = train_df['answer'][1]

print(f"\nQuestion\n")
print(f"{question}\n")
print("------------------------------------------------------------------------")
#print(f"\nContext\n")
#print(f"{context}\n")
print("------------------------------------------------------------------------")
print(f"\nAnswer\n\n")
print(answer)

In [ ]:
# Count NaN / null values per column for train set
train_null_counts = train_df.isna().sum()

# Count NaN / null values per column for test set
test_null_counts = test_df.isna().sum()

null_counts = train_null_counts + test_null_counts
# Print the result
print(f"NaN/null values per column: \n{null_counts}")
print(f"NaN/null values per column in train: \n{train_null_counts}")
print(f"NaN/null values per column in test: \n{test_null_counts}")


train_df = train_df.dropna()
test_df = test_df.dropna()

print('Any missing values: ')
print(train_df.isnull().any() & test_df.isnull().any())

### Tokenize

Let's tokenize our text before running analysis on it

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
from nltk.stem import PorterStemmer

nltk.download('punkt')
nltk.download('stopwords')



stop_words = set(stopwords.words('english'))
# stemmer = PorterStemmer()

context_lengths = []
question_lengths = []
vocab = set()

all_tokens = []

train_df_tokenized = []

def preprocess(text):
    tokens = word_tokenize(text.lower())
    tokens = [word for word in tokens if word.isalnum()]  # Remove punctuation
    tokens = [word for word in tokens if word not in stop_words]  # Remove stopwords
    # tokens = [stemmer.stem(word) for word in tokens]  # Apply stemming
    return tokens

def append_tokens(entry):
    context_tokens = preprocess(entry['context'])
    question_tokens = preprocess(entry['question'])
    answer_tokens = preprocess(entry['answer'])
    
    context_lengths.append(len(context_tokens))
    question_lengths.append(len(question_tokens))
    vocab.update(context_tokens)
    vocab.update(question_tokens)

    all_tokens.extend(context_tokens)
    all_tokens.extend(question_tokens)

    train_df_tokenized.append({'context': context_tokens, 'question': question_tokens, 'answer': answer_tokens})


train_df.apply(lambda entry: append_tokens(entry), axis=1)

In [ ]:
train_df_tokenized = pd.DataFrame(train_df_tokenized, columns=['context', 'question', 'answer'])
train_df_tokenized.head()

### Analysis of Token Frequencies (TF-IDF)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

train_df_tokenized['context_str'] = train_df_tokenized['context'].apply(lambda tokens: ' '.join(tokens))
train_df_tokenized['question_str'] = train_df_tokenized['question'].apply(lambda tokens: ' '.join(tokens))

# TF-IDF on contexts only
vectorizer_context = TfidfVectorizer()
tfidf_context = vectorizer_context.fit_transform(train_df_tokenized['context_str'])

# TF-IDF on questions only
vectorizer_question = TfidfVectorizer()
tfidf_question = vectorizer_question.fit_transform(train_df_tokenized['question_str'])

# TF-IDF on combined contexts + questions
train_df_tokenized['combined'] = train_df_tokenized['context_str'] + ' ' + train_df_tokenized['question_str']
vectorizer_combined = TfidfVectorizer()
tfidf_combined = vectorizer_combined.fit_transform(train_df_tokenized['combined'])

We can see that the tf-idf matrix is much thinner than the question one. This is natural since the context has a much bigger vocabulary. 
\
Another interesting thing to point out is that the matrix due to the combination of both question and context is slightly wider than the context one because most of the words contained in the questions are also contained in the context.

In [ ]:
tfidf_question_df = pd.DataFrame(tfidf_question.toarray(), columns=vectorizer_question.get_feature_names_out())
tfidf_question_df

In [ ]:
tfidf_context_df = pd.DataFrame(tfidf_context.toarray(), columns=vectorizer_context.get_feature_names_out())
tfidf_context_df

In [ ]:
tfidf_combined_df = pd.DataFrame(tfidf_combined.toarray(), columns=vectorizer_combined.get_feature_names_out())
tfidf_combined_df

Let's visualize through a histogram the most frequent word in the contexts

In [ ]:
word_scores = tfidf_context_df.sum(axis=0).sort_values(ascending=False)

top_n = 20
top_words = word_scores.head(top_n)

plt.figure(figsize=(10, 6))
top_words.plot(kind='bar')
plt.title(f"Top {top_n} Words by TF-IDF Score")
plt.xlabel("Words")
plt.ylabel("Total TF-IDF Score")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
feature_names = vectorizer_context.get_feature_names_out()

def get_top_n_words(row, n):
    row_data = row.toarray().flatten()
    top_indices = row_data.argsort()[::-1][:n]
    
    return [(feature_names[i], row_data[i]) for i in top_indices if row_data[i] > 0]

N = 5
train_df_tokenized[f'top_{N}_tfidf_words_context'] = [
    get_top_n_words(tfidf_context[i], N) for i in range(tfidf_context.shape[0])
]

In [ ]:
train_df_tokenized.head()

Let's see the most frequent words (calculated through  tf-idf score) for each context

In [ ]:
def plot_top_words_with_scores(word_score_pairs, title='Top TF-IDF Words'):
    words, scores = zip(*word_score_pairs)
    plt.figure(figsize=(5, 4))
    plt.barh(words[::-1], scores[::-1], color='red')  # reverse for descending order
    plt.xlabel('TF-IDF Score')
    plt.title(title)
    plt.tight_layout()
    plt.show()

for i in range(10):
    plot_top_words_with_scores(train_df_tokenized.loc[i, f'top_{N}_tfidf_words_context'],
                           title=f'Document {i+1} - Top TF-IDF Words')

Now instead let's visualize the most frequent tokens contained in the train set including all of the columns (question, context, answer)

In [ ]:
fdist = FreqDist(all_tokens)
fdist

In [ ]:
from wordcloud import WordCloud

wordcloud = WordCloud(width=800, height=400).generate_from_frequencies(fdist)
plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.title("Word Cloud of Vocabulary")
plt.show()

In [ ]:
fdist.plot(30,  title="Top 30 Most Frequent Words")

There are lots of words that appear just once. Let's count them

In [ ]:
least_freq_1_words = [word for word, freq in fdist.items() if freq == 1 and word.isalpha()]
print(len(least_freq_1_words))

These are some of the words that appear only once 

Now we'd like to produce some statistics on the length of the contexts and questions

In [ ]:
print(f"Number of documents: {len(train_df)}")
print(f"Average context length: {sum(context_lengths)/len(context_lengths):.2f} tokens")
print(f"Average question length: {sum(question_lengths)/len(question_lengths):.2f} tokens")
print(f"Vocabulary size: {len(vocab)}")

sns.histplot(context_lengths, bins=50, kde=True)
plt.title('Distribution of Context Lengths')
plt.xlabel('Number of Tokens')
plt.ylabel('Frequency')
plt.show()

In [ ]:
sns.histplot(question_lengths, bins=50, kde=True)
plt.title("Question Length Distribution")
plt.xlabel("Number of Tokens")
plt.ylabel("Frequency")
plt.show()

## 1.b) Data cleaning

* Stemming? necessary or not

In [ ]:
tokenized_contexts = train_df_tokenized['context']
tokenized_questions = train_df_tokenized['question']
tokenized_answers = train_df_tokenized['answer']

print(len(tokenized_questions))
print(len(tokenized_contexts))
print(len(tokenized_answers))

# Optionally filter out short sentences (more than 3 tokens)
#tokenized_questions = tokenized_questions[tokenized_questions.apply(lambda x: len(x) > 3)]
#tokenized_contexts = tokenized_contexts[tokenized_contexts.apply(lambda x: len(x) > 3)]
#tokenized_answers = tokenized_answers[tokenized_answers.apply(lambda x: len(x) > 3)]

print(len(tokenized_questions))
print(len(tokenized_contexts))
print(len(tokenized_answers))

Create the vector to feed the Word2Vec model in such a way to have (question, context, answer). \
There are other possibilities to form the vector to feed to the model (e. g. the one commented out ) they all perform pretty well and more or less the same

In [ ]:
all_tokenized_sentences = [ c + q + a for c, q, a in zip(tokenized_contexts.tolist(), tokenized_questions.tolist(), tokenized_answers.tolist()) ]

# all_tokenized_sentences = tokenized_contexts.tolist() + tokenized_questions.tolist() + tokenized_answers.tolist() # ther possibility performing worse

### Drop the words occurring once

Convert least_freq_1_words to a set because it is much faster to search in it and then filter out all the words that occurr once (N.T. all the words not the numbers)

In [ ]:
least_freq_1_words = set(least_freq_1_words)
all_tokenized_sentences = [ [ word for word in sentence if word not in least_freq_1_words ] for sentence in all_tokenized_sentences ]


### Create embeddings with Word2Vec 



In [ ]:
from gensim.models.word2vec import Word2Vec

d = 30 # dimension of the embeddings

model = Word2Vec(all_tokenized_sentences, vector_size=d, min_count=5, window=10) # default model is skipgram (it tends to perform better)

In [ ]:
len(model.wv)

# print(model.wv.key_to_index) # output is too long

In [ ]:
term = 'house'
model.wv[term]
model.wv.most_similar(term)

In [ ]:
seed = random.seed(0)

sample = random.sample(list(model.wv.key_to_index), 1000)
word_vectors = model.wv[sample]
word_vectors

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=3, n_iter=2000)
tsne_embedding = tsne.fit_transform(word_vectors)

In [ ]:
x, y, z = np.transpose(tsne_embedding)

In [ ]:
fig = px.scatter_3d(x=x, y=y, z=z)
fig.update_traces(marker=dict(size=3,line=dict(width=2)))
fig.show()

In [ ]:
fig = px.scatter_3d(x=x[:200],y=y[:200],z=z[:200],text=sample[:200])
fig.update_traces(marker=dict(size=3,line=dict(width=2)),textfont_size=10)
fig.show()

In [ ]:
colours = ['red','green','blue','orange','yellow','purple','pink','cream','brown','black','white','gray']

word_vectors = model.wv[colours+sample]

tsne = TSNE(n_components=3)
tsne_embedding = tsne.fit_transform(word_vectors)

x, y, z = np.transpose(tsne_embedding)

In [ ]:
r = (-200,200)
fig = px.scatter_3d(x=x, y=y, z=z, range_x=r, range_y=r, range_z=r, text=colours + [None] * 1000)
fig.update_traces(marker=dict(size=3,line=dict(width=2)),textfont_size=10)
fig.show()

### 1.c) Clustering

Now that we have the embeddings of the words, we would like to make operations fast with them. \
Faiss is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning. \
Faiss is written in C++ with complete wrappers for Python/numpy. Some of the most useful algorithms are implemented on the GPU. It is developed primarily at Meta's Fundamental AI Research group. \
Let's install it and try to make a search!

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss
index = faiss.IndexFlatL2(d) 

In [ ]:
# This creates a matrix with the embeddings of the words as rows
words = list(model.wv.key_to_index.keys())
word_vectors = np.array([model.wv[word] for word in words]).astype("float32") 

# This creates an index 
index = faiss.IndexFlatL2(word_vectors.shape[1])
index.add(word_vectors)

In [ ]:
# Let's perform a search
query_word = 'house'
query_vector = np.array([model.wv[query_word]]).astype("float32")
D, I = index.search(query_vector, k=10)  # D = distances, I = indices
similar_words = [words[i] for i in I[0]]
print(f"Words similar to '{query_word}':", similar_words)

Let's create a function that shows examples of clusters

In [ ]:
def show_cluster_examples(column, n_clusters=5, top_n=30):
    if column == "question": 
        df = pd.DataFrame({column: tokenized_questions, 'cluster': labels})   
    elif column == "context": 
        df = pd.DataFrame({column: tokenized_context, 'cluster': labels})
    elif column == "answer": 
        df = pd.DataFrame({column: tokenized_answers, 'cluster': labels})
    
    unique_clusters = sorted(df['cluster'].unique()) # Get unique labels and sort them
    clusters_to_show = unique_clusters[:n_clusters]

    for i in clusters_to_show:
        print(f"\n Cluster {i} (size={len(df[df.cluster==i])}):")
        examples = df[df.cluster == i][column].sample(min(top_n, len(df[df.cluster == i])), random_state=0)
        for q in examples:
            print("  -", q)

Let's now use the embeddings of the words provided by Word2Vec. \
The Word2Vec gives us the embeddings of words, now we have to compute the embeddings of sentences to then compare them and cluster them. 
We use the mean of the embedding of the words to form the embeddings of the sentence (N.T. We could have also used the sum).

In [ ]:
def sentence_embedding(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

question_embeddings = np.array([sentence_embedding(tokens, model) for tokens in tokenized_questions])
context_embeddings = np.array([sentence_embedding(tokens, model) for tokens in tokenized_contexts])
answer_embeddings = np.array([sentence_embedding(tokens, model) for tokens in tokenized_answers])

This is a function to guess the magnitude of k. It works in this way, we calculate the cosine similarity of the sentences embeddings and count them 

In [ ]:
from sklearn.cluster import KMeans

def cosine_similarity_faiss(vectors):
    vectors = np.asarray(vectors).astype('float32')
    normalized = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
    similarity_matrix = np.matmul(normalized, normalized.T) 
    return similarity_matrix

def extract_similar_pairs(sim_matrix, threshold):
    n = sim_matrix.shape[0]
    similar_pairs = []

    for i in range(1, n):
        for j in range(i):
            if sim_matrix[i, j] > threshold:
                similar_pairs.append((i, j, sim_matrix[i, j]))
    l = len(similar_pairs)
    print(f"Sim pairs: {l}")
    return l

def sim_matrix_density(embeddings, threshold=0.8):
    S = cosine_similarity_faiss(embeddings)
    print(S.shape)
    l = extract_similar_pairs(S, threshold)
    return l/(S.shape[0] * S.shape[1])*100, S

The idea of having the sim_matrix_density function is to have an idea of which percentage of the sentences is similar. \
In this way it is possible to get a small indicator of the number of the topics. \
The bigger the percentage the less the topics

In [ ]:
perc, S = sim_matrix_density(question_embeddings, 0.7) 

In [ ]:
print(f"The percentage of similar entries of the similarity matrix is {perc:.2f}%")

In [ ]:
S = cosine_similarity_faiss(question_embeddings)

upper_triangle = S[np.triu_indices_from(S)]

plt.hist(upper_triangle, bins=50, edgecolor='black')
plt.title('Distribution of Cosine Similarity Scores')
plt.xlabel('Cosine Similarity')
plt.ylabel('Frequency')
plt.show()

### Elbow method

In [ ]:
k_values = range(80, 500, 10)
wcss = []

for k in tqdm(k_values):
    kmeans = KMeans(n_clusters=k, random_state=seed, n_init=10)
    kmeans.fit(question_embeddings)
    wcss.append(kmeans.inertia_)

plt.plot(k_values, wcss, marker='o')
plt.xlabel("Number of Clusters (k)")
plt.ylabel("WCSS (Inertia)")
plt.title("Elbow Method on Sentence Embeddings")
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.xticks(range(min(k_values), max(k_values) + 1, 10))
plt.show()

In [ ]:
k = 250

In [ ]:
# 250 - 300 working fine
kmeans = KMeans(n_clusters=k, random_state=0)
labels = kmeans.fit_predict(question_embeddings)

In [ ]:
show_cluster_examples('question')

In [ ]:
kmeans = KMeans(n_clusters=k, random_state=0)
labels = kmeans.fit_predict(answer_embeddings)

In [ ]:
show_cluster_examples('answer')

### Let's now use a Transformer to create the embeddings 

In [ ]:
print(tokenized_questions.tolist()[-10:])

In [ ]:
tokenized_questions = tokenized_questions[tokenized_questions.apply(lambda x: len(x) > 3)]
tokenized_contexts = tokenized_contexts[tokenized_contexts.apply(lambda x: len(x) > 3)]
tokenized_answers = tokenized_answers[tokenized_answers.apply(lambda x: len(x) > 3)]

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2") 
question_embeddings = model.encode(tokenized_questions.tolist(), show_progress_bar=True)

In [ ]:
#k, S = guess_k_for_clustering(question_embeddings, 0.99)
print(k)

In [ ]:
upper_triangle = S[np.triu_indices_from(S)]

plt.hist(upper_triangle, bins=50, edgecolor='black')
plt.title('Distribution of Cosine Similarity Scores')
plt.xlabel('Cosine Similarity')
plt.ylabel('Frequency')
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=300, random_state=0)
labels = kmeans.fit_predict(question_embeddings)

In [ ]:
show_cluster_examples('question', 10)

In [ ]:
show_cluster_examples('answer')

In [ ]:
#show_cluster_examples('context')

In [ ]:
answer_embeddings = model.encode(tokenized_answers.tolist(), show_progress_bar=True)


In [ ]:
kmeans = KMeans(n_clusters=300, random_state=0)
labels = kmeans.fit_predict(answer_embeddings)

In [ ]:
show_cluster_examples('answer')

# Training models

## GPU acceleration with CUDA if possible

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # accelerate on GPU if available

## Claudia
### Sottotitolo 1
Inserisci qui il tuo testo

## Bert 

For this part we won't use our tokenizer but the one from the library

In [ ]:
from transformers import AutoTokenizer, BertForNextSentencePrediction

bert_model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
bert = BertForNextSentencePrediction.from_pretrained("bert-base-uncased", output_attentions=True) # we need output_attentions = True for visualization 

n_params = sum(param.numel() for param in bert.parameters())
print(f"This model has {n_params} paramaters")

In [ ]:
bert = bert.to(device)

In [ ]:
def print_all_bert_parameters():
    for name, param in bert.named_parameters():
        print(f"Parameter name: {name}")
        print(f"Parameter shape: {param.size()}")
        print(f"Is trainable: {param.requires_grad}")
        print()

In [ ]:
# print_all_bert_parameters() # Commented out because output  is too long

We have the:
1. Embeddings layer
- word embeddings (30522 x 768)
- positional embeddings (512 x 768)
- token type (2 X 768)
- layer norm (768 x 768)
2. 12x Encoders layers each one including
- Multi-Head Self-Attention Block [...]
- Feed-Forward block
3. Pooling layer
- Poooler dense weights
- Pooler dense biases

In [ ]:
print("vocabulary size: ", len(tokenizer.vocab))


In [ ]:
def evaluate_answers_through_BERT(train_df, max_length):

    label_tensor = torch.LongTensor([1]).to(device)
    
    count = 0
    correct = 0
    
    train_df = train_df[:max_length]

    for row in train_df.itertuples():
        if row.context and row.question and row.answer:
            text_a = row.context + " " + row.question
            text_b = row.answer
            
            encoding = tokenizer(text_a, text_b, return_tensors="pt", max_length=512, truncation=True).to(device)

            with torch.no_grad():
                outputs = bert(**encoding, labels=label_tensor)
                logits = outputs.logits
                print(f"{count} {logits}")
                if logits[0][0].item() > logits[0][1].item():
                    correct += 1

            count += 1

    return correct / count if count > 0 else 0


In [ ]:
correct_answers = evaluate_answers_through_BERT(train_df, 100) # only evaluated on 100 samples otherwise it is really slow
print(f"Percentage of correct answers: {correct_answers * 100} %")

### Attention visualization through BERT (this is the only part needing a patch)

In [ ]:
!pip install -q bertviz

In [ ]:
from bertviz.neuron_view import show

i=2
q = train_df['question'][i] 
a = train_df['answer'][i]
print(q)
print(a)

model_type = 'bert'
head_view(bert, model_type, tokenizer, q, a)



In [ ]:
from bertviz.transformers_neuron_view import BertModel, BertTokenizer
from bertviz.neuron_view import show
model_type = 'bert'
model_version = 'bert-base-uncased'
do_lower_case = True
model = BertModel.from_pretrained(model_version)
tokenizer = BertTokenizer.from_pretrained(model_version, do_lower_case=do_lower_case)
sentence_a = "The cat sat on the mat"
sentence_b = "The cat lay on the rug"
show(model, model_type, tokenizer, q, a, display_mode='dark', layer=2, head=0)

### Semantic Search with RoBerta

In [ ]:
from sentence_transformers import SentenceTransformer, util
roberta = SentenceTransformer('all-distilroberta-v1')

In [ ]:
question_roberta_embeddings = roberta.encode(train_df['question'], convert_to_tensor=True)
# context_roberta_embeddings = roberta.encode(train_df['context'], convert_to_tensor=True) # Commented out because it takes too long to run
answer_roberta_embeddings = roberta.encode(train_df['answer'], convert_to_tensor=True)

normalized_question_embeddings = util.normalize_embeddings(question_roberta_embeddings)
# normalized_context_embeddings = util.normalize_embeddings(context_roberta_embeddings) 
normalized_answer_embeddings = util.normalize_embeddings(answer_roberta_embeddings)

In [ ]:
hits = util.semantic_search(normalized_question_embeddings, normalized_context_embeddings, score_function=util.dot_score)

In [ ]:
queries = ['What are the main benefits of having available clean water resources?']

top_k = min(5, len(question_roberta_embeddings))
for query in queries:
    query_embedding = roberta.encode(query, convert_to_tensor=True)
    cos_scores = util.cos_sim(query_embedding, question_roberta_embeddings)[0]
    top_results = torch.topk(cos_scores, k=top_k)
    print(f"Query: \"{query}\"")
    print("Top 5 most similar sentences in corpus:\n\n")
    for score, idx in zip(top_results[0], top_results[1]):
        idx = idx.item()  # Convert tensor to int
        print(f"Score: {score:.2f} - Document: \"{train_df['question'].iloc[idx]}\"")
    print("\n\n\n\n")


## Alberto

**Importing the dataset and removing Nulls**

In [15]:
from datasets import load_dataset

dataset = load_dataset("neural-bridge/rag-dataset-12000")

# Define a function to keep only valid examples
def is_valid(example):
    return all([
        example["question"] is not None and example["question"].strip() != "",
        example["context"] is not None and example["context"].strip() != "",
        example["answer"] is not None and example["answer"].strip() != ""
    ])

# Apply filtering
dataset = dataset.filter(is_valid)

README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

(…)-00000-of-00001-9df3a936e1f63191.parquet:   0%|          | 0.00/23.1M [00:00<?, ?B/s]

(…)-00000-of-00001-af2a9f454ad1b8a3.parquet:   0%|          | 0.00/5.79M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9600 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2400 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9600 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2400 [00:00<?, ? examples/s]

**Defining dataset and dataset loader classes for LSTM**

In [10]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

class QADataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        input_text = f"question: {item['question']} context: {item['context']}"
        target_text = item['answer']

        input_ids = self.tokenizer.encode(input_text, truncation=True, padding="max_length", max_length=self.max_length)
        target_ids = self.tokenizer.encode(target_text, truncation=True, padding="max_length", max_length=64)

        return {
            "input_text": input_text,
            "target_text": target_text,
            "input_ids": torch.tensor(input_ids),
            "target_ids": torch.tensor(target_ids)
        }

In [16]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")  # can be any tokenizer with .encode/.pad capabilities
max_length = 512

train_dataset = QADataset(dataset['train'], tokenizer, max_length)
test_dataset = QADataset(dataset['test'], tokenizer, max_length)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=True)

**Defining LSTM model**

In [12]:
import torch
import torch.nn as nn

class Seq2SeqLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(Seq2SeqLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.encoder = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.decoder = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_ids, target_ids):
        embedded_inputs = self.embedding(input_ids)
        _, (hidden, cell) = self.encoder(embedded_inputs)

        embedded_targets = self.embedding(target_ids)
        outputs, _ = self.decoder(embedded_targets, (hidden, cell))

        logits = self.fc(outputs)
        return logits

**Train loop**

The following loop is optional, another option is to simply load the model

In [13]:
def train_model(model, criterion, optimizer, vocab_size, device, trian_loader):
    for epoch in range(200):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader):
            input_ids = batch["input_ids"].to(device)
            target_ids = batch["target_ids"].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids, target_ids[:, :-1])
            loss = criterion(outputs.reshape(-1, vocab_size), target_ids[:, 1:].reshape(-1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
    
        print(f"Epoch {epoch+1} - Loss: {total_loss / len(train_loader)}")

    torch.save(model, 'model.pt')

In [19]:
from tqdm import tqdm

TRIANING = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if (TRIANING):
    print('Model trainin start: ')
    vocab_size = tokenizer.vocab_size
    criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    model = Seq2SeqLSTM(vocab_size, embed_size=256, hidden_size=512).to(device)

    train_model(model, criterion, optimizer, vocab_size, device, train_loader)
else:
    print('Loading model:')
    model = torch.load('/kaggle/input/lstm-pretrained-model/pytorch/default/1/model.pt')
    print(model)

Loading model:


/tmp/ipykernel_31/1699561833.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load('/kaggle/input/lstm-pretrained-model/pytorch/default/1/model.pt')


In [25]:
def eval_model(model, scorer, device, test_loader):
    model.eval()

    rouge1_scores = []
    rougeL_scores = []
    em_scores = []
    bleu_scores = []
    
    texts = []
    predictions = []
    references = []
    
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(device)
        target_ids = batch["target_ids"].to(device)
    
        # Inference: greedy decoding
        with torch.no_grad():
            embedded_inputs = model.embedding(input_ids)
            _, (hidden, cell) = model.encoder(embedded_inputs)
    
            decoder_input = target_ids[:, 0].unsqueeze(1)  # first token (e.g. <start>)
            outputs = []
    
            # predicts all characters in the sentence
            for t in range(1, target_ids.shape[1]):
                embedded = model.embedding(decoder_input)
                output, (hidden, cell) = model.decoder(embedded, (hidden, cell))
                logits = model.fc(output.squeeze(1))
                predicted = logits.argmax(1)
                outputs.append(predicted.unsqueeze(1))
                decoder_input = predicted.unsqueeze(1)
    
            outputs = torch.cat(outputs, dim=1)  # (batch_size, seq_len - 1)
    
        texts += batch['input_text']
        # Convert outputs to text
        for pred_ids, true_ids in zip(outputs, target_ids[:, 1:]):
            pred_tokens = [token.item() for token in pred_ids if token.item() != tokenizer.pad_token_id]
            true_tokens = [token.item() for token in true_ids if token.item() != tokenizer.pad_token_id]
    
            pred_text = tokenizer.decode(pred_tokens, skip_special_tokens=True).strip()
            true_text = tokenizer.decode(true_tokens, skip_special_tokens=True).strip()
    
            references.append(true_text)
            predictions.append(pred_text)
    
            # Exact Match
            em = int(pred_text == true_text)
            em_scores.append(em)
    
            # BLEU
            ref = [true_text.split()]
            hyp = pred_text.split()
            bleu = sentence_bleu(ref, hyp, weights=(0.5, 0.5))  # BLEU-2
            bleu_scores.append(bleu)
    
            # ROUGE
            rouge = scorer.score(true_text, pred_text)
            rouge1_scores.append(rouge["rouge1"].fmeasure)
            rougeL_scores.append(rouge["rougeL"].fmeasure)

    return rouge1_scores, rougeL_scores, em_scores, bleu_scores, texts, predictions, references

In [22]:
!pip install evaluate rouge_score

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 6.9 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=2da2115895867834a3cfac4a58c68dd10c9fb8dbe1a5404c7f66e4f813a60076
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.12.0 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_syst

In [26]:
from nltk.translate.bleu_score import sentence_bleu
from torch.nn.functional import softmax
from rouge_score import rouge_scorer


scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
rouge1_scores, rougeL_scores, em_scores, bleu_scores, texts, predictions, references = eval_model(model, scorer, device, test_loader)

100%|██████████| 150/150 [00:26<00:00,  5.63it/s]


In [27]:
print(f"Exact Match (EM): {sum(em_scores)/len(em_scores):.4f}")
print(f"BLEU Score: {sum(bleu_scores)/len(bleu_scores):.4f}")
print(f"Rouge1: {sum(rouge1_scores)/len(rouge1_scores):.4f}")
print(f"RougeL: {sum(rougeL_scores)/len(rougeL_scores):.4f}")

Exact Match (EM): 0.0000
BLEU Score: 0.0153
Rouge1: 0.1370
RougeL: 0.1043


#### LLM evaluation

In [28]:
def construct_prompt(example, context=True, few_shot=False, shots=[]):
    prompt = ""
    if few_shot:
        for ex in shots:
            prompt += f"question: {ex['question']}\n"
            if context:
                prompt += f"context: {ex['context']}\n"
            prompt += f"answer: {ex['answer']}\n\n"
    prompt += f"question: {example['question']}\n"
    if context:
        prompt += f"context: {example['context']}"
    return prompt.strip()

In [35]:
def evaluate_llm(test_set, train_set, scorer, context=True, few_shot=False, k=0, max_eval=50):
    predictions = []
    references = []
    rouge1_scores = []
    rougeL_scores = []
    
    shot_examples = train_set.select(range(k)) if few_shot and k > 0 else []

    for i, example in enumerate(test_set.select(range(max_eval))):  # limit for speed
        shots = [shot_examples[j] for j in range(k) if j != i] if few_shot else []
        prompt = construct_prompt(example, context=context, few_shot=few_shot, shots=shots)

        try:
            output = generator(prompt, max_length=64, clean_up_tokenization_spaces=True)[0]['generated_text'].strip()
        except Exception as e:
            print(f"Skipping example {i} due to error: {e}")
            continue

        predictions.append(output)
        references.append(example["answer"].strip())
        rouge = scorer.score(example["answer"].strip(), output)
        rouge1_scores.append(rouge["rouge1"].fmeasure)
        rougeL_scores.append(rouge["rougeL"].fmeasure)
    
    return rouge1_scores, rougeL_scores

In [31]:
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer
import evaluate

model_checkpoint = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
generator = pipeline("text2text-generation", model=model, tokenizer=tokenizer)

2025-05-19 08:59:32.130729: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747645172.309666      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747645172.364189      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cuda:0


In [36]:
test_set = dataset['test']
train_set = dataset['train']


# ---------- Run All 4 Evaluation Modes ----------
print("\n🧪 Evaluating all 4 QA modes with ROUGE (first 50 examples):\n")

print("No-context Zero-shot")
rouge1_scores, rougeL_scores = evaluate_llm(test_set,train_set, scorer, context=False, few_shot=False, max_eval=20)
print(f"Rouge1: {sum(rouge1_scores)/len(rouge1_scores):.4f}")
print(f"RougeL: {sum(rougeL_scores)/len(rougeL_scores):.4f}")


print("\nNo-context Few-shot")
rouge1_scores, rougeL_scores =evaluate_llm(test_set,train_set,scorer, context=False, few_shot=True, k=2, max_eval=20)
print(f"Rouge1: {sum(rouge1_scores)/len(rouge1_scores):.4f}")
print(f"RougeL: {sum(rougeL_scores)/len(rougeL_scores):.4f}")


print("\nWith-context Zero-shot")
rouge1_scores, rougeL_scores = evaluate_llm(test_set,train_set,scorer, context=True, few_shot=False, max_eval=20)
print(f"Rouge1: {sum(rouge1_scores)/len(rouge1_scores):.4f}")
print(f"RougeL: {sum(rougeL_scores)/len(rougeL_scores):.4f}")


print("\nWith-context Few-shot")
rouge1_scores, rougeL_scores = evaluate_llm(test_set,train_set,scorer, context=True, few_shot=True, k=2, max_eval=20)
print(f"Rouge1: {sum(rouge1_scores)/len(rouge1_scores):.4f}")
print(f"RougeL: {sum(rougeL_scores)/len(rougeL_scores):.4f}")


🧪 Evaluating all 4 QA modes with ROUGE (first 50 examples):

No-context Zero-shot
Rouge1: 0.0378
RougeL: 0.0321

No-context Few-shot


Token indices sequence length is longer than the specified maximum sequence length for this model (1109 > 512). Running this sequence through the model will result in indexing errors


Rouge1: 0.2329
RougeL: 0.2070

With-context Zero-shot
Rouge1: 0.3808
RougeL: 0.3688

With-context Few-shot
Rouge1: 0.4186
RougeL: 0.4006


## Martina

# Extensions

## Chatbot

We created a web application using Streamlit that is available at https://chatbot-m4btjtwos4j.streamlit.app/. \
The web app is divided into 2 sections. \
The first section contains a chatbot built on top of Meta's model Llama 3.1 8B Instruct which is a general purpose chatbot. \
The second section contains a button that randomly chooses a question and answer pair from our dataset that sends the question through an API request and then displays the answer from the model side by side with the answer stored in the dataset.


# Conclusions